# 02 Context and State

So far we have given our agents access to a lot of information & functionality via:
* **Prompts** -> system prompts via the `system_prompt` parameter of the `create_agent()` function and user-prompts when we `invoke()` the agent using the `{"messages" : {"role": "user", "context": query}]}` parameter to `invoke()`
* **Tools** -> custom tools we created and passed in via the `tools=[...]` parameter of the `create_agent()` call.
* **Chat Memory** -> via an instance of `langgraph.checkpoint.memory.InMemorySaver` class passed via the `checkpointer` parameter of `create_agent()` call
* **MCP Servers** -> Access to additional 3rd party tools via MCP Servers

What if you wanted to pass in additional _pre-defined_ information in the context that the agent needs to know, which it can share with it's tools? For example, let's suppose you are developing a Text-to-SQL agent, which converts your text query to a SQL and executes it against your database to fetch the rows. Such an Agent would need access to a database connection and a relevant schema to convert the text to SQL. Now let's see how that will work:

**Step1: define the structure of the context info**

First you must define a `dataclass` that will hold all the pre-populated information you want to pass into the agent [you can also use a `TypedDict` instead of the dataclass]. We'll cover both the techniques. This additional context is actually not useful to the agent, but is used within the tools you define for the agent.

In [1]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In [1]:
from langchain_community.utilities import SQLDatabase

# NOTE: here I am creating a database connection to a database
# and fetching the database schema - for this example I am using a
# SQLite database (Chinook) saved locally in the db sub-folder, but
# this could be ANY database - an enterprise Oracle/SQL-Server/PostgreSQL etc.

db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
schema = db.get_table_info()

In [2]:
# NOTE: comment out one of the 2 blocks below to use either the dataclas or TypedDict version

from dataclasses import dataclass


@dataclass
class RuntimeContext:
    # the database connection
    db: SQLDatabase
    # the schema
    db_schema: str


# from typing import TypedDict

# class RuntimeContext(TypedDict):
#     # the database connection
#     db: SQLDatabase
#     # the schema
#     db_schema: str

Next, define the tool for our agent. This tool executes the query generated by the LLM
against the database, hence the database connection will be required by the tool. The `langgraph.runtime.get_runtime()` call helps any tool function access the pre-populated context passed to the agent in any tool call. This is how we use it inside our tool function.

In [7]:
# Step 2 - define the tool for our agent
from langchain.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """executes SQL query against the database"""

    # if you defined the context as a dataclass
    # use the following line
    db: SQLDatabase = get_runtime().context.db

    # if you define the context as a TypedDict,
    # use the following line instead (since TypedDict is a dict)
    # db: SQLDatabase = get_runtime().context["db"]
    # Use ONLY one of the 2 mechanisms above, not both!

    try:
        # now you can use db (SQLDatabase) to
        # run your query as usual
        result = db.run(query)
    except Exception as e:
        return f"Error occurred while executing SQL query: {e}"
    return str(result)

Define your agent as below 

In [8]:
from langchain.agents import create_agent

system_prompt = """You are a careful SQLite Analyst.
    Rules:
    - Always think step-by-step
    - When you need data, call the tool 'execute_sql' with ONE select query
    - Read-only only; NO INSERT/UPDATE/DELETE/DROP/CREATE/REPLACE/TRUNCATE/ALTER
    - Limit to 5 rows at the output, unless the user explicitly asks for more
    - If the tool returns "Error:", revise the SQL and try again
    - Prefer explicit column list, avoid SELECT *
    - Here is the database schema you can refer to to generate the SQL
    {database_schema}
"""

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=system_prompt,
    tools=[execute_sql],
    # here is where you tell the agent the
    # datatype of the pre-populated context
    context_schema=RuntimeContext,
)

Finally, `invoke()` agent with your text query. Here is where you'll create an instance of your context class and pass it into the `invoke()` function. 

In [10]:
user_query = "List all my customers living in the city of London or Paris"

# create an instance of RuntimeContext & pass into invoke
# as follows
runtime_context = RuntimeContext(db=db, db_schema=schema)

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    context=runtime_context,
)

# display final result (should be the results of our query)
print(response["messages"][-1].content)

Here are the customers in London or Paris (up to 5 rows):
- CustomerId 39 — Camille Bernard — Paris
- CustomerId 40 — Dominique Lefebvre — Paris
- CustomerId 52 — Emma Jones — London
- CustomerId 53 — Phil Hughes — London

Would you like me to include additional fields (e.g., Company, Address) or show all matching records if there are more?


To view the intermediate steps, which will also show you the SQL `select XXXX` statement, run the following code block instead.

In [11]:
user_query = "List all my customers living in the city of London or Paris"

# create an instance of RuntimeContext & pass into invoke
# as follows
runtime_context = RuntimeContext(db=db, db_schema=schema)

for step in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=runtime_context,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

List all my customers living in the city of London or Paris
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_0EKlgrjIIKbMh2U41E7uYE8Q)
 Call ID: call_0EKlgrjIIKbMh2U41E7uYE8Q
  Args:
    query: SELECT customer_id, first_name, last_name, city FROM customers WHERE lower(city) IN ('london','paris') LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error occurred while executing SQL query: (sqlite3.OperationalError) no such column: customer_id
[SQL: SELECT customer_id, first_name, last_name, city FROM customers WHERE lower(city) IN ('london','paris') LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_l0S6ZTpF0ee0iaIVQoWFlx3E)
 Call ID: call_l0S6ZTpF0ee0iaIVQ

So this is how we can pass around pre-defined context to our Agent (tools), which they can _read_ but not modify. 

### Updateable context

What if we want to pass around something that the agent can _update_? If you recall, this is possible using the `InMemorySaver()` class and passed in to the `create_agent()` via its `checkpointer=InMemoryState()` parameter. However, the agent automatically updates this context with the list of messages exchanged so far in the conversation context. You have no control over this process.

We can also add our own _custom fields_ to this context as shown in the example below:

1. We have to define an instance of the `langchain.agents.AgentState` class, which hold our custom fields - this is similar to a dataclass we have see so far. This is how it works.


In [27]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In [28]:
# Step 1: define the structure of the AgentState
from langchain.agents import AgentState


# An AgentState is a dict that can hold any number of fields
# which are to be updated by the agent
class CustomState(AgentState):
    favourite_color: str
    least_favourite_color: str

In [29]:
# Step2: define tools, include special tool functions that update the AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


# this is a special tool function that updates the members of the AgentState
# it is usually the first function called by the Agent based on user-query
@tool
def update_favourite_colors(
    favourite_color: str,
    least_favourite_color: str,
    runtime: ToolRuntime,
) -> Command:
    """update the favourite and least favourite colors of user once they have revealed them"""
    print(
        f" -------- update_favourite_colors(favourite_color={favourite_color}, least_favourite_color={least_favourite_color}) tool called --------"
    )

    print("\n ----- Runtime ------ ")
    console.print(runtime)

    return Command(
        update={
            # below are key-value pairs, where the keys match the names of fields you
            # defined in the AgentState class above & values are values you want to set
            # for those fields - typically "parsed from user query by the LLM"
            "favourite_color": favourite_color,
            "least_favourite_color": least_favourite_color,
            "messages": [
                ToolMessage(
                    f"Successfully updated colors: favourite -> {favourite_color} and least favourite -> {least_favourite_color}",
                    tool_call_id=runtime.tool_call_id,
                ),
            ],
        }
    )


# other tool functions to read-agent state & do other work
@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- get_favourite_color({runtime.state}) tool called --------")
    print(" ---- runtime state ---- ")
    console.print(runtime)
    return runtime.state["favourite_color"]


@tool
def get_least_favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- get_least_favourite_color({runtime.state}) tool called --------")
    print(" ---- runtime state ---- ")
    console.print(runtime)
    return runtime.state["least_favourite_color"]

Let's create the agent - do it as usual.

Now let's call the agent. Note that the first query I pass into the agent tells it my favourite & last favourite colors, which should trigger the `update_favourite_colors()` tool. The other 2 queries will trigger the `get_favourite_color()` and `get_least_favourite_color()` tools.

In [30]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# color_pref = ColorContext(favourite_color="blue", least_favourite_color="yellow")
state_schema = CustomState(favourite_color="blue", least_favourite_color="yellow")

agent2 = create_agent(
    model="openai:gpt-5-nano",
    # system_prompt=(
    #     """
    #     You are a helpful agent that can answer information about my color preference.
    #     You have access to the following tools:
    #         - get_favourite_color : to get my favourite color
    #         - get_least_favourite_color : to get my least favourite color
    #         - update_favourite_colors : to update favourite & least favourite colors of user
    #     Use these tools to set color my preferences and respond to any questions regarding
    #     my color preference & return response as
    #         "Your [least]favourite color is <<color>>"
    #     For all other queries about me, respond as follows:
    #         "Sorry, this information is confidential. I can only tell you about your color preference"
    #     """
    # ),
    tools=[get_favourite_color, get_least_favourite_color, update_favourite_colors],
    checkpointer=InMemorySaver(),
    # here you tell the agent the structure of your state schema
    state_schema=CustomState,
)

In [31]:
# helper function
from typing import Any


def ask_agent(agent, query, config) -> Any:
    response = agent.invoke(
        {"messages": {"role": "user", "content": query}},
        config=config,
    )
    return response


def ask_agent2(agent, query, config):
    for step in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        config=config,
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

In [32]:
user_query = "My favourite color is blue and least favourite is yellow"

agent_config = {"configurable": {"thread_id": "1234"}}

response = ask_agent(agent2, user_query, agent_config)
print(response["messages"][-1].content)

 -------- update_favourite_colors(favourite_color=blue, least_favourite_color=yellow) tool called --------

 ----- Runtime ------ 


ToolRuntime(
    state={
        'messages': [
            HumanMessage(
                content='My favourite color is blue and least favourite is yellow',
                additional_kwargs={},
                response_metadata={},
                id='8f3e7f97-efa8-4dd1-8ae8-64890134c36a'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 290,
                        'prompt_tokens': 187,
                        'total_tokens': 477,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 256,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5-nano-2025-08-07',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-DX5PSNfaVD8uZ64LR44L2O3jOghHJ',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019db038-8a6c-78d3-9906-9120940039ee-0',
                tool_calls=[
                    {
                        'name': 'update_favourite_colors',
                        'args': {'favourite_color': 'blue', 'least_favourite_color': 'yellow'},
                        'id': 'call_R5cnAVGGaQ5NwxcDbPzYownH',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 187,
                    'output_tokens': 290,
                    'total_tokens': 477,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 256}
                }
            )
        ]
    },
    context=None,
    config={
        'tags': [],
        'metadata': {
            'ls_integration': 'langchain_create_agent',
            'thread_id': '1234',
            'langgraph_step': 2,
            'langgraph_node': 'tools',
            'langgraph_triggers': ('__pregel_push',),
            'langgraph_path': ('__pregel_push', 0, False),
            'langgraph_checkpoint_ns': 'tools:0ddb81c4-2c24-b9ae-f934-d6b6a23dd2f9',
            'checkpoint_ns': 'tools:0ddb81c4-2c24-b9ae-f934-d6b6a23dd2f9'
        },
        'callbacks': <langchain_core.callbacks.manager.CallbackManager object at 0x00000207817762A0>,
        'recursion_limit': 9999,
        'configurable': {
            'thread_id': '1234',
            '__pregel_runtime': Runtime(
                context=None,
                store=None,
                stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000020782841800>,
                previous=None,
                execution_info=ExecutionInfo(node_attempt=1, node_first_attempt_time=1776777995.1660972)
            ),
            '__pregel_replay_state': None,
            '__pregel_task_id': '0ddb81c4-2c24-b9ae-f934-d6b6a23dd2f9',
            '__pregel_send': <built-in method extend of collections.deque object at 0x00000207819A31F0>,
            '__pregel_read': functools.partial(<function local_read at 0x00000207FE4EBB00>, 
PregelScratchpad(step=2, stop=9999, call_counter=<langgraph.pregel._algo.LazyAtomicCounter object at 
0x00000207817776D0>, interrupt_counter=<langgraph.pregel._algo.LazyAtomicCounter object at 0x00000207828BA5F0>, 
get_null_resume=<function _scratchpad.<locals>.get_null_resume at 0x0000020782841440>, resume=[], 
subgraph_counter=<langgraph.pregel._algo.LazyAtomicCounter object at 0x0000020

Done. I've updated your preferences: favorite color is blue and least favorite color is yellow. Would you like to update or check them again, or add anything else?


In [33]:
user_query2 = "What's my favourite color?"

response = ask_agent(agent2, user_query2, agent_config)
print(response["messages"][-1].content)

 -------- get_favourite_color({'messages': [HumanMessage(content='My favourite color is blue and least favourite is yellow', additional_kwargs={}, response_metadata={}, id='8f3e7f97-efa8-4dd1-8ae8-64890134c36a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 290, 'prompt_tokens': 187, 'total_tokens': 477, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DX5PSNfaVD8uZ64LR44L2O3jOghHJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db038-8a6c-78d3-9906-9120940039ee-0', tool_calls=[{'name': 'update_favourite_colors', 'args': {'favourite_color': 'blue', 'least_favourite_color': 'yellow'}, 'id': 'call_R5cnAVGGaQ5

ToolRuntime(
    state={
        'messages': [
            HumanMessage(
                content='My favourite color is blue and least favourite is yellow',
                additional_kwargs={},
                response_metadata={},
                id='8f3e7f97-efa8-4dd1-8ae8-64890134c36a'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 290,
                        'prompt_tokens': 187,
                        'total_tokens': 477,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 256,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5-nano-2025-08-07',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-DX5PSNfaVD8uZ64LR44L2O3jOghHJ',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019db038-8a6c-78d3-9906-9120940039ee-0',
                tool_calls=[
                    {
                        'name': 'update_favourite_colors',
                        'args': {'favourite_color': 'blue', 'least_favourite_color': 'yellow'},
                        'id': 'call_R5cnAVGGaQ5NwxcDbPzYownH',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 187,
                    'output_tokens': 290,
                    'total_tokens': 477,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 256}
                }
            ),
            ToolMessage(
                content='Successfully updated colors: favourite -> blue and least favourite -> yellow',
                name='update_favourite_colors',
                id='2b63d926-1972-4966-b154-f56ce0a20890',
                tool_call_id='call_R5cnAVGGaQ5NwxcDbPzYownH'
            ),
            AIMessage(
                content="Done. I've updated your preferences: favorite color is blue and least favorite color is 
yellow. Would you like to update or check them again, or add anything else?",
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 426,
                        'prompt_tokens': 240,
                        'total_tokens': 666,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 384,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5-nano-2025-08-07',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-DX5PXfSBX9o8joT8bEY5qdNUTNGJO',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019db038-a3c3-79b3-a07b-577dcbcba10c-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_toke

Your favourite color is blue. Would you like to check or update any other preferences?


In [34]:
user_query3 = "What's my least favourite color?"

response = ask_agent(agent2, user_query3, agent_config)
print(response["messages"][-1].content)

 -------- get_least_favourite_color({'messages': [HumanMessage(content='My favourite color is blue and least favourite is yellow', additional_kwargs={}, response_metadata={}, id='8f3e7f97-efa8-4dd1-8ae8-64890134c36a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 290, 'prompt_tokens': 187, 'total_tokens': 477, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DX5PSNfaVD8uZ64LR44L2O3jOghHJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db038-8a6c-78d3-9906-9120940039ee-0', tool_calls=[{'name': 'update_favourite_colors', 'args': {'favourite_color': 'blue', 'least_favourite_color': 'yellow'}, 'id': 'call_R5cnA

ToolRuntime(
    state={
        'messages': [
            HumanMessage(
                content='My favourite color is blue and least favourite is yellow',
                additional_kwargs={},
                response_metadata={},
                id='8f3e7f97-efa8-4dd1-8ae8-64890134c36a'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 290,
                        'prompt_tokens': 187,
                        'total_tokens': 477,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 256,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5-nano-2025-08-07',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-DX5PSNfaVD8uZ64LR44L2O3jOghHJ',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019db038-8a6c-78d3-9906-9120940039ee-0',
                tool_calls=[
                    {
                        'name': 'update_favourite_colors',
                        'args': {'favourite_color': 'blue', 'least_favourite_color': 'yellow'},
                        'id': 'call_R5cnAVGGaQ5NwxcDbPzYownH',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 187,
                    'output_tokens': 290,
                    'total_tokens': 477,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 256}
                }
            ),
            ToolMessage(
                content='Successfully updated colors: favourite -> blue and least favourite -> yellow',
                name='update_favourite_colors',
                id='2b63d926-1972-4966-b154-f56ce0a20890',
                tool_call_id='call_R5cnAVGGaQ5NwxcDbPzYownH'
            ),
            AIMessage(
                content="Done. I've updated your preferences: favorite color is blue and least favorite color is 
yellow. Would you like to update or check them again, or add anything else?",
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 426,
                        'prompt_tokens': 240,
                        'total_tokens': 666,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 384,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5-nano-2025-08-07',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-DX5PXfSBX9o8joT8bEY5qdNUTNGJO',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019db038-a3c3-79b3-a07b-577dcbcba10c-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_toke

Your least favourite color is yellow. Would you like to check or update any other preferences?
